# ModelEdge — QLoRA Fine-tuning on Google Colab T4

This notebook runs the complete fine-tuning pipeline on a free Colab T4 GPU.

**Steps:**
1. Install dependencies
2. Connect Weights & Biases
3. Download + prepare MedQA dataset
4. Fine-tune with QLoRA + Unsloth
5. Save adapter weights to Google Drive

In [ ]:
# Verify GPU
!nvidia-smi

In [ ]:
# Install core dependencies
!pip install -q unsloth[colab-new] peft accelerate datasets bitsandbytes trl wandb
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

In [ ]:
import wandb
wandb.login()  # Paste your W&B API key when prompted

In [ ]:
# Mount Google Drive for saving checkpoints
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/modeledge/finetuned'
import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Clone the repository
!git clone https://github.com/sakshiasati17/ModelEdge.git /content/ModelEdge
%cd /content/ModelEdge

In [ ]:
# Prepare the MedQA dataset
!python data/prepare_dataset.py --dataset medqa --output data/processed/

In [ ]:
# Quick check — inspect a few training samples
import json

with open('data/processed/medqa/train.jsonl') as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        print(f'--- Sample {i+1} ---')
        print(rec['text'][:400])
        print()
        if i >= 2:
            break

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048
MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print('Model loaded. Parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset as hf_load_dataset
import json

def load_jsonl(path):
    recs = []
    with open(path) as f:
        for line in f:
            if line.strip():
                recs.append(json.loads(line))
    return recs

from datasets import Dataset
train_ds = Dataset.from_list(load_jsonl('data/processed/medqa/train.jsonl'))
val_ds   = Dataset.from_list(load_jsonl('data/processed/medqa/val.jsonl'))
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
import wandb
wandb.init(project='modeledge-medqa', name='qlora-llama3.2-3b-colab')

from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    fp16=True,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='wandb',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
)

In [ ]:
trainer.train()
wandb.finish()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

In [ ]:
# Quick inference test
FastLanguageModel.for_inference(model)

test_prompt = (
    'Below is a medical question. Answer it accurately and concisely.\n\n'
    '### Instruction:\nWhat is the first-line treatment for hypertension?\n\n'
    '### Input:\n\n'
    '### Response:\n'
)

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
generated = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))